# Section 01 lab — Spark foundations

Run the section setup once. Every exercise creates or reads the data it needs and can then be run independently.


## Section setup — run once


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
print('Spark:', spark.version, '| master:', spark.sparkContext.master)


## Exercise 1 — Create and verify a Spark session


In [ ]:
print('Spark:', spark.version)
print('Master:', spark.sparkContext.master)
print('Application:', spark.sparkContext.appName)


## Exercise 2 — Create a DataFrame from Python rows and inspect partitions


In [ ]:
from pyspark.sql import functions as F

employees = [(1, 'Engineering', 90000), (2, 'Sales', 72000),
             (3, 'Engineering', 96000), (4, 'Finance', 81000),
             (5, 'Sales', 76000), (6, 'Finance', 83000)]
employees_df = spark.createDataFrame(employees, ['employee_id', 'department', 'salary']).repartition(2)
employees_df.withColumn('partition_id', F.spark_partition_id()).orderBy('employee_id').show()
print('Partitions:', employees_df.rdd.getNumPartitions())


## Exercise 3 — Distinguish transformations from actions


In [ ]:
from pyspark.sql import functions as F

numbers = spark.range(0, 20, 1, 4)
even_squares = (numbers.filter(F.col('id') % 2 == 0)
                .withColumn('square', F.col('id') * F.col('id')))
print('The filter and derived column are transformations.')
even_squares.explain('formatted')
print('Row count:', even_squares.count())
even_squares.orderBy('id').show()
assert even_squares.count() == 10


## Exercise 4 — Compare repartition and coalesce


In [ ]:
from pyspark.sql import functions as F

rows = [(1, 'A'), (2, 'B'), (3, 'A'), (4, 'C'), (5, 'B'), (6, 'A')]
base = spark.createDataFrame(rows, ['id', 'category'])
four_partitions = base.repartition(4, 'category')
two_partitions = four_partitions.coalesce(2)
print('Original partitions:', base.rdd.getNumPartitions())
print('After repartition:', four_partitions.rdd.getNumPartitions())
print('After coalesce:', two_partitions.rdd.getNumPartitions())
four_partitions.withColumn('partition_id', F.spark_partition_id()).orderBy('id').show()


## Exercise 5 — Connect an action to its execution plan


In [ ]:
from pyspark.sql import functions as F

events = spark.range(0, 40).withColumn('group_id', F.col('id') % 4)
summary = events.groupBy('group_id').agg(F.count('*').alias('rows'))
summary.explain('formatted')
summary.orderBy('group_id').show()
assert summary.count() == 4
